# Notebook 3 — Narrowing the task instead of enlarging the model

Notebooks 1 and 2 vary the model. This one leaves the model alone and varies **how the request
is written**, which is the lever an organization controls without changing any infrastructure.

**Block A — specifying the output.** Eight narrow tasks, each asked five ways: with a plain
instruction or with a written output contract (a role, a strict format and two worked
examples), on a small, a medium and a large model. The question is whether a specification
lets a smaller model do the same work.

**Block B — splitting the request.** Three composite requests, each handled two ways: one call
to the large model, or split into three sub-questions whose answers a final call assembles.

Independent of notebooks 1 and 2. Allow about 45 minutes.

In [ ]:
import random
import re

import numpy as np
import pandas as pd

from core import ask, save_table, SEED

SMALL, MEDIUM, LARGE = "qwen2.5:0.5b", "qwen2.5:1.5b", "qwen2.5:7b"

ROLE = ("Tu es un service d'extraction. Réponds UNIQUEMENT par le résultat demandé, "
        "sans phrase d'introduction, sans explication, sans ponctuation superflue.")

EXAMPLES = ("Exemple 1 — Entrée : « Facture du 03/02/2024 » -> 03/02/2024\n"
            "Exemple 2 — Entrée : « Montant : 1 250,00 € » -> 1250,00")


def with_contract(task_text):
    """A role, a strict output format, and two worked examples, in front of the task."""
    return f"{ROLE}\n\n{EXAMPLES}\n\nTÂCHE :\n{task_text}"


CONDITIONS = {
    "small_plain":     {"model": SMALL,  "contract": False},
    "small_contract":  {"model": SMALL,  "contract": True},
    "medium_contract": {"model": MEDIUM, "contract": True},
    "large_plain":     {"model": LARGE,  "contract": False},
    "large_contract":  {"model": LARGE,  "contract": True},
}

print("five conditions:", ", ".join(CONDITIONS))

## 1. The eight tasks of Block A

Four families, two tasks each. Every task carries a deterministic check, so no judge is used
anywhere in this notebook — these are narrow tasks with one right answer, which is precisely
why they are the ones worth specifying.

> **On comparability with the earlier campaign.** The eight task identifiers and families below
> are the ones used previously, but the task *text* was never stored in the results file, so the
> sentences here are a reconstruction: same structure, different wording. Replacing the `task`
> fields with the original strings is the only change needed to make Block A comparable run for
> run; nothing else in the notebook depends on them.

In [ ]:
def simplify(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()


def digits_only(text):
    return re.sub(r"[^0-9+]", "", str(text))


TASKS_A = [
    {"id": "A1_extraction_date", "family": "Extraction",
     "task": "Extrais la date de : « Facture n°88 émise le 14/05/2025 ». "
             "Réponds uniquement par la date.",
     "check": lambda answer: int("14/05/2025" in str(answer))},

    {"id": "A2_extraction_montant", "family": "Extraction",
     "task": "Extrais le montant de : « Total à régler : 2 480,50 € TTC ». "
             "Réponds uniquement par le nombre.",
     "check": lambda answer: int("248050" in digits_only(answer))},

    {"id": "A3_classification_technique", "family": "Classification",
     "task": "Classe ce message dans Facturation, Technique ou Commercial : "
             "« L'application plante au démarrage ». Réponds uniquement par la catégorie.",
     "check": lambda answer: int("technique" in simplify(answer)
                                 and "facturation" not in simplify(answer))},

    {"id": "A4_classification_facturation", "family": "Classification",
     "task": "Classe ce message dans Facturation, Technique ou Commercial : "
             "« Ma facture de mars est fausse ». Réponds uniquement par la catégorie.",
     "check": lambda answer: int("facturation" in simplify(answer)
                                 and "technique" not in simplify(answer))},

    {"id": "A5_traduction_reunion", "family": "Translation",
     "task": "Traduis en anglais, sans rien ajouter : « La réunion est reportée à mardi. »",
     "check": lambda answer: int("meeting" in simplify(answer)
                                 and "tuesday" in simplify(answer))},

    {"id": "A6_traduction_livraison", "family": "Translation",
     "task": "Traduis en anglais, sans rien ajouter : "
             "« La livraison est retardée d'une semaine. »",
     "check": lambda answer: int("deliver" in simplify(answer)
                                 and ("delayed" in simplify(answer)
                                      or "late" in simplify(answer)))},

    {"id": "A7_normalisation_date", "family": "Normalisation",
     "task": "Normalise cette date au format AAAA-MM-JJ : « 7 mars 2025 ». "
             "Réponds uniquement par la date.",
     "check": lambda answer: int("2025-03-07" in str(answer))},

    {"id": "A8_normalisation_numero", "family": "Normalisation",
     "task": "Normalise ce numéro au format international français : « 06 12 34 56 78 ». "
             "Réponds uniquement par le numéro.",
     "check": lambda answer: int("+33612345678" in digits_only(answer))},
]

print(f"{len(TASKS_A)} tasks, "
      f"{len(set(task['family'] for task in TASKS_A))} families, "
      f"{len(TASKS_A) * len(CONDITIONS)} runs")

## 2. Run Block A

Two precautions, and the first campaign showed what happens without them.

**The model is unloaded before every call**, so each condition pays its own loading cost.
**The order of the five conditions is shuffled within each task**, with a fixed seed, so no
condition is systematically the one that runs on a machine already warmed up. In the first
campaign two conditions ran on an already-loaded model and looked far cheaper than they were,
which reversed the sign of the whole result. Section 3 checks that this has not happened
again.

In [ ]:
rows = []

for task_index, task in enumerate(TASKS_A):
    order = list(CONDITIONS)
    random.Random(SEED + task_index).shuffle(order)

    for condition_name in order:
        condition = CONDITIONS[condition_name]
        prompt = with_contract(task["task"]) if condition["contract"] else task["task"]
        result = ask(condition["model"], prompt, max_new_tokens=120)
        correct = task["check"](result["answer"])

        rows.append({
            "task": task["id"], "family": task["family"], "condition": condition_name,
            "model": condition["model"], "contract": condition["contract"],
            "energy_wh": result["energy_wh"], "duration_s": result["duration_s"],
            "load_s": result["load_s"], "tokens_out": result["tokens_out"],
            "correct": correct, "answer": result["answer"][:300],
        })

        print(f"{task['id']:30} {condition_name:17} {result['energy_wh']:7.4f} Wh   "
              f"load {result['load_s']:5.2f} s   "
              f"{'correct' if correct else 'wrong'}")

block_a = save_table(pd.DataFrame(rows), "blocA_runs")

## 3. Check the measurement before reading it

Every condition should show a real model-loading time, rising with model size. A condition near
zero found the model already in memory and did not pay what the others paid, which makes the
energy figures incomparable.

In [ ]:
loading = block_a.groupby("condition").load_s.mean().sort_values()
print("mean model-loading time per condition, seconds")
print(loading.round(2).to_string())

if loading.min() < 0.2:
    print("\nPROBLEM: at least one condition never loaded the model.")
    print("Its energy is understated against the others. Do not use these figures.")
    print("Delete outputs/blocA_runs.csv and run section 2 again.")
else:
    print("\nFine: every condition paid its own model load.")

## 4. Block A — the result

Three comparisons answer the question. What does the contract alone buy on a small model, and
on a large one? And can a medium model with a contract match a large model without one?

In [ ]:
summary = block_a.groupby("condition").agg(
    runs=("correct", "size"),
    energy_wh=("energy_wh", "mean"),
    accuracy=("correct", "mean"),
    tokens_out=("tokens_out", "mean"))
summary["accuracy %"] = 100 * summary.accuracy
save_table(summary.reset_index().round(4), "blocA_summary")
print(summary[["runs", "energy_wh", "accuracy %", "tokens_out"]].round(4).to_string())


def compare(first, second, label):
    left = block_a[block_a.condition == first]
    right = block_a[block_a.condition == second]
    energy_change = 100 * (right.energy_wh.mean() - left.energy_wh.mean()) / left.energy_wh.mean()
    print(f"{label:48} energy {energy_change:+7.1f} %    "
          f"accuracy {100 * left.correct.mean():5.1f} % -> {100 * right.correct.mean():5.1f} %")


print()
compare("small_plain", "small_contract", "the contract alone, on the small model")
compare("large_plain", "large_contract", "the contract alone, on the large model")
compare("large_plain", "medium_contract", "large without contract -> medium with contract")

print("\naccuracy by family, %")
print((100 * block_a.pivot_table(index="family", columns="condition",
                                 values="correct", aggfunc="mean")).round(0).to_string())

## 5. Block B — splitting a composite request

Three requests that each ask for several things at once. Each is answered twice: once in a
single call to the large model, and once split into three sub-questions whose answers a final
call assembles into the required format.

The assembly call is measured separately, because it is where most of the cost of the split
pipeline turns out to sit.

In [ ]:
ASSEMBLY_PROMPT = "Assemble ces éléments en un seul JSON valide, sans rien ajouter :\n{parts}"

INVOICE = "« Facture Lumen SAS du 12/03/2025, 1 250,00 € TTC, abonnement logiciel »"
TICKET = "« Impossible de se connecter depuis la mise à jour, 40 utilisateurs bloqués »"
CONTACT = "« dupont jean / 06 12 34 56 78 / Jean.DUPONT@exemple.FR »"

TASKS_B = [
    {"id": "B1", "label": "Invoice triage",
     "whole": f"À partir de cette ligne, donne la date, le montant et la catégorie comptable, "
              f"en JSON : {INVOICE}.",
     "parts": [f"Donne uniquement la date de : {INVOICE}",
               f"Donne uniquement le montant de : {INVOICE}",
               f"Donne uniquement la catégorie comptable de : {INVOICE}"],
     "check": lambda answer: int("12/03/2025" in str(answer)
                                 and "125000" in digits_only(answer))},

    {"id": "B2", "label": "Ticket routing",
     "whole": f"Pour ce ticket, donne la catégorie, l'urgence (haute, moyenne ou basse) "
              f"et l'équipe, en JSON : {TICKET}.",
     "parts": [f"Catégorie de ce ticket, parmi Facturation, Technique, Commercial : {TICKET}",
               f"Urgence de ce ticket, parmi haute, moyenne, basse : {TICKET}",
               f"Équipe à saisir pour ce ticket : {TICKET}"],
     "check": lambda answer: int("technique" in simplify(answer)
                                 and "haute" in simplify(answer))},

    {"id": "B3", "label": "Contact normalisation",
     "whole": f"Normalise ce contact en JSON avec les clés nom, telephone, email : {CONTACT}.",
     "parts": [f"Mets le nom en forme « Prénom Nom » : {CONTACT}",
               f"Mets le numéro au format international français : {CONTACT}",
               f"Mets l'email en minuscules : {CONTACT}"],
     "check": lambda answer: int("+33612345678" in digits_only(answer)
                                 and "jean.dupont@exemple.fr" in simplify(answer))},
]

print(f"{len(TASKS_B)} composite requests")
print("B2's urgency check expects « haute »: that is a judgement the reference answer fixes,")
print("not a fact, so read B2 as the weakest of the three checks.")

In [ ]:
rows = []

for task in TASKS_B:
    # ---- one call to the large model
    whole = ask(LARGE, task["whole"])
    rows.append({
        "task": task["id"], "label": task["label"], "arm": "monolithic", "calls": 1,
        "energy_wh": whole["energy_wh"], "duration_s": whole["duration_s"],
        "energy_assembly_wh": 0.0, "tokens_out": whole["tokens_out"],
        "correct": task["check"](whole["answer"]), "answer": whole["answer"][:300],
    })

    # ---- three sub-questions, then one assembly call
    energy = 0.0
    duration = 0.0
    partial_answers = []
    for part in task["parts"]:
        piece = ask(LARGE, part, max_new_tokens=80)
        energy += piece["energy_wh"]
        duration += piece["duration_s"]
        partial_answers.append(piece["answer"].strip())

    assembly = ask(LARGE, ASSEMBLY_PROMPT.format(parts="\n".join(partial_answers)),
                   max_new_tokens=200)
    rows.append({
        "task": task["id"], "label": task["label"], "arm": "decomposed",
        "calls": len(task["parts"]) + 1,
        "energy_wh": energy + assembly["energy_wh"],
        "duration_s": duration + assembly["duration_s"],
        "energy_assembly_wh": assembly["energy_wh"], "tokens_out": assembly["tokens_out"],
        "correct": task["check"](assembly["answer"]), "answer": assembly["answer"][:300],
    })

    print(f"{task['id']}  monolithic {rows[-2]['energy_wh']:7.4f} Wh   "
          f"decomposed {rows[-1]['energy_wh']:7.4f} Wh   "
          f"(assembly alone {assembly['energy_wh']:.4f} Wh)")

block_b = save_table(pd.DataFrame(rows), "blocB_runs")

In [ ]:
summary_b = block_b.groupby("arm").agg(
    requests=("correct", "size"),
    energy_wh=("energy_wh", "mean"),
    calls=("calls", "mean"),
    correct=("correct", "sum"))
print(summary_b.round(4).to_string())

decomposed = block_b[block_b.arm == "decomposed"]
assembly_share = 100 * decomposed.energy_assembly_wh.sum() / decomposed.energy_wh.sum()
print(f"\nthe assembly call alone is {assembly_share:.0f} % of the split pipeline's energy")
print("Three requests: this is a direction and a mechanism, not an effect size.")

## 6. Figure

Cost against accuracy, one point per condition. The useful reading is vertical distance for a
given horizontal position: two conditions at the same energy with different accuracy, or the
same accuracy at different energy.

In [ ]:
from core import MODEL_COLOURS, new_figure, save_figure

ORDER = ["small_plain", "small_contract", "medium_contract", "large_plain", "large_contract"]
points = (block_a.groupby("condition")
                 .agg(energy_wh=("energy_wh", "mean"),
                      accuracy=("correct", "mean"),
                      model=("model", "first"))
                 .reindex(ORDER))

figure, axes = new_figure(
    "Cost against accuracy, by condition",
    "Eight narrow tasks with deterministic checks. Darker marks are larger models; "
    "a ring marks the conditions that carry an output contract.",
    "Accuracy (%)", width=10, height=6.4)

for condition in ORDER:
    row = points.loc[condition]
    axes.scatter(row.energy_wh, 100 * row.accuracy, s=230,
                 color=MODEL_COLOURS[row.model], zorder=3,
                 edgecolor="#1A1A1A" if condition.endswith("contract") else "none",
                 linewidth=1.6)
    axes.annotate(condition.replace("_", " "), (row.energy_wh, 100 * row.accuracy),
                  xytext=(12, 8), textcoords="offset points", fontsize=10, color="#5A6472")

axes.set_xlabel("Energy per request (Wh)", fontsize=10, color="#5A6472")
axes.set_xlim(0, points.energy_wh.max() * 1.35)
axes.set_ylim(-5, 105)
axes.grid(axis="x", color="#E4E4E1", linewidth=0.8, zorder=0)
save_figure(figure, axes, "F7_specialisation")

## 7. Output of this notebook

In `outputs/`:

| File | What it holds |
|---|---|
| `blocA_runs.csv` | one row per task and condition: 40 rows |
| `blocA_summary.csv` | the five conditions compared |
| `blocB_runs.csv` | one row per request and arm: 6 rows |
| `figures/F7_specialisation.png` | cost against accuracy |